# 04 — Live Signal (Daily Use)

Run this notebook each trading day.

- **Cell 1 (run anytime after 3:10 PM):** Authenticate and fetch latest data
- **Cell 2:** Check filters (India VIX, S&P 500)
- **Cell 3:** Compute signal, display buy list, save positions
- **Cell 4 (run at 9:25 AM next morning):** Fetch open prices, compute overnight P&L
- **Cell 5:** Update trade log

## Cell 1 — Authenticate & fetch data

If token is stale, run `generate_and_save_token()` first.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from datetime import date
import pandas as pd

from config import NIFTY50_SYMBOLS, POSITIONS_FILE, CAPITAL, WEIGHT_PER_STOCK, TOP_N
from utils import (
    get_kite_session,
    get_nifty50_instruments,
    load_all_ohlc,
    compute_overnight_returns,
    compute_scores,
    rank_stocks,
    check_us_filter,
    check_vix_filter,
    compute_transaction_cost,
    append_trade_log,
    rolling_performance_check,
)

TODAY = date.today().isoformat()
print(f'Date: {TODAY}')

# Uncomment if token is stale:
# from utils import generate_and_save_token
# kite = generate_and_save_token()

kite = get_kite_session()
print('Kite authenticated.')

ohlc_dict    = load_all_ohlc(NIFTY50_SYMBOLS)
overnight_df = compute_overnight_returns(ohlc_dict)
scores_df    = compute_scores(overnight_df)
print(f'Data loaded. Latest date: {overnight_df.index[-1].date()}')

## Cell 2 — Check filters

In [ ]:
us_pass, us_ret = check_us_filter(TODAY)
vix_pass, vix_val = check_vix_filter(kite)

us_icon  = '✓' if us_pass  else '✗'
vix_icon = '✓' if vix_pass else '✗'

us_label  = f'S&P 500 prev: {us_ret:+.2%}'  if not pd.isna(us_ret)  else 'S&P 500: N/A'
vix_label = f'India VIX: {vix_val:.1f}'      if not pd.isna(vix_val) else 'India VIX: N/A'

print(f'Filter: {us_label} {us_icon} | {vix_label} {vix_icon}')

TRADE_TODAY = us_pass and vix_pass
if TRADE_TODAY:
    print('\n→ All filters PASS. Proceed to Cell 3.')
else:
    print('\n→ Filter(s) FAILED. Do NOT trade today.')
    if not us_pass:
        print(f'  Reason: S&P 500 was negative ({us_ret:+.2%})')
    if not vix_pass:
        print(f'  Reason: India VIX ({vix_val:.1f}) > 20')

## Cell 3 — Compute signal & buy list

Run between 3:10–3:20 PM. Place orders between 3:20–3:25 PM.

In [ ]:
if not TRADE_TODAY:
    print('Filters failed — no trades today.')
else:
    signal_date = scores_df.index[-1]
    buy_list = rank_stocks(scores_df, signal_date, top_n=TOP_N)
    scores_row = scores_df.loc[signal_date]

    # Get prev close prices
    close_prices = {}
    for sym in buy_list:
        df = ohlc_dict.get(sym)
        if df is not None and not df.empty:
            avail = df.index[df.index <= signal_date]
            if not avail.empty:
                close_prices[sym] = float(df.loc[avail[-1], 'close'])

    # Display
    print(f'\n=== BUY LIST — {TODAY} (enter at 3:20–3:25 PM) ===')
    print(f'{"Rank":<6} {"Symbol":<14} {"Score":>8}  {"Prev Close":>12}  {"Qty":>6}  {"Rs":>10}')
    print('-' * 64)

    positions = []
    for rank, sym in enumerate(buy_list, 1):
        score  = scores_row.get(sym, float('nan'))
        prev_c = close_prices.get(sym, None)
        qty    = int(WEIGHT_PER_STOCK / prev_c) if prev_c else 0
        score_s = f'{score:+.4%}' if not pd.isna(score) else 'N/A'
        close_s = f'{prev_c:,.2f}' if prev_c else 'N/A'
        print(f'{rank:<6} {sym:<14} {score_s:>8}  {close_s:>12}  {qty:>6}  {WEIGHT_PER_STOCK:>10,.0f}')
        positions.append({
            'symbol': sym, 'rank': rank,
            'signal_score': float(score) if not pd.isna(score) else None,
            'prev_close': prev_c,
            'allocated_rs': WEIGHT_PER_STOCK,
            'qty': qty,
            'entry_price': None,
            'date': TODAY,
        })

    print(f'\nTotal to deploy: Rs {CAPITAL:,.0f}')
    print('Order type: CNC (delivery)')

    # Save positions
    POSITIONS_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(POSITIONS_FILE, 'w') as f:
        json.dump(positions, f, indent=2)
    print(f'\nPositions saved → {POSITIONS_FILE}')
    print('After filling orders, manually update entry_price in that file.')

## Cell 4 — Morning exit (9:25 AM next day)

Run at 9:25 AM after the open has settled.

In [ ]:
import time
from config import KITE_RATE_LIMIT_SLEEP

if not POSITIONS_FILE.exists():
    print(f'No positions file: {POSITIONS_FILE}')
else:
    with open(POSITIONS_FILE) as f:
        positions = json.load(f)

    instruments = get_nifty50_instruments()

    print(f'=== SELL SIGNAL — {date.today()} 09:25 AM ===')
    print(f'{"Symbol":<14} {"Entry":>10} {"Exit":>10} {"Overnight":>10}  {"P&L (Rs)":>10}')
    print('-' * 62)

    total_pnl = 0.0
    total_inv = 0.0
    trade_records = []

    for pos in positions:
        sym         = pos['symbol']
        entry_price = pos.get('entry_price')
        qty         = pos.get('qty', 0)

        if entry_price is None:
            print(f'{sym:<14} entry_price is None — update {POSITIONS_FILE}')
            continue

        try:
            quote = kite.quote([f'NSE:{sym}'])
            ohlc  = quote[f'NSE:{sym}']['ohlc']
            exit_price = float(ohlc.get('open') or quote[f'NSE:{sym}']['last_price'])
        except Exception as e:
            print(f'{sym:<14} ERROR: {e}')
            continue

        overnight_ret = exit_price / entry_price - 1
        gross_pnl     = (exit_price - entry_price) * qty
        cost          = compute_transaction_cost(entry_price, qty)
        net_pnl       = gross_pnl - cost

        total_pnl += net_pnl
        total_inv += entry_price * qty

        print(f'{sym:<14} {entry_price:>10,.2f} {exit_price:>10,.2f} {overnight_ret:>+10.2%}  {net_pnl:>+10.0f}')
        time.sleep(KITE_RATE_LIMIT_SLEEP)

        trade_records.append({
            'date': date.today().isoformat(),
            'symbol': sym, 'entry_price': entry_price, 'exit_price': exit_price,
            'qty': qty, 'overnight_return_pct': overnight_ret,
            'pnl_rs': gross_pnl, 'transaction_cost_rs': cost, 'net_pnl_rs': net_pnl,
            'signal_score': pos.get('signal_score'), 'rank': pos.get('rank'),
            'vix_at_entry': None, 'us_prev_close_return': None, 'session_notes': '',
        })

    print('-' * 62)
    tot_ret = total_pnl / total_inv if total_inv else float('nan')
    print(f'\nTOTAL NET P&L: Rs {total_pnl:+,.0f}')
    print(f'TOTAL RETURN:  {tot_ret:+.2%}' if not pd.isna(tot_ret) else 'TOTAL RETURN: N/A')

## Cell 5 — Update trade log & rolling check

In [ ]:
if trade_records:
    append_trade_log(trade_records)

print()
rolling_performance_check()